In [0]:
# Spark Query Optimization — Interview Critical!
from pyspark.sql import functions as F

# Create large-ish dataset
data = [(i, f"model_{i%10}",
         float(70 + i%30), i%5)
        for i in range(10000)]
df = spark.createDataFrame(
    data, ["id", "model", "accuracy", "team"]
)

# 1. AVOID: Multiple scans
bad = df.filter(F.col("accuracy") > 85)
bad2 = df.filter(F.col("team") == 1)
# Scans df TWICE

# BETTER: Combine filters
good = df.filter(
    (F.col("accuracy") > 85) &
    (F.col("team") == 1)
)  # Single scan!

# 2. Cache frequently used DataFrames
# Note: .cache() not supported on serverless compute

# 3. Repartition for better parallelism
df_repartitioned = df.repartition(4, "team")
# Note: .rdd access not supported on serverless compute

# 4. Broadcast join — small table optimization
small_df = spark.createDataFrame(
    [(0,"TeamA"),(1,"TeamB"),
     (2,"TeamC"),(3,"TeamD"),(4,"TeamE")],
    ["team_id", "team_name"]
)
# Broadcast small_df to all workers
from pyspark.sql.functions import broadcast
result = df.join(
    broadcast(small_df),
    df.team == small_df.team_id
)
result.explain()  # see broadcast in plan!

# 5. Explain plan — understand execution
df.filter(F.col("accuracy") > 90)\
  .groupBy("model")\
  .agg(F.avg("accuracy"))\
  .explain(True)  # True = verbose plan

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonBroadcastHashJoin [team#11066L], [team_id#11067L], Inner, BuildRight, false, true
         :- PhotonRowToColumnar
         :  +- LocalTableScan [id#11063L, model#11064, accuracy#11065, team#11066L]
         +- PhotonShuffleExchangeSource
            +- PhotonShuffleMapStage EXECUTOR_BROADCAST, [id=#7002]
               +- PhotonShuffleExchangeSink SinglePartition
                  +- PhotonRowToColumnar
                     +- LocalTableScan [team_id#11067L, team_name#11068]


== Photon Explanation ==
The query is fully supported by Photon.
== Parsed Logical Plan ==
'Aggregate ['model], ['model, unresolvedalias('avg('accuracy))]
+- 'Filter '`>`('accuracy, 90)
   +- 'UnresolvedSubqueryColumnAliases [id, model, accuracy, team]
      +- LocalRelation [id#11043L, model#11044, accuracy#11045, team#11046L]

== Analyzed Logical Plan ==
model: string, avg(